In [ ]:
import sys
import os
import json
import torch
import pandas as pd
import random
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import napari

# %%
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f'using Device: {device}')

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

sys.path.append(f"{project_root}src/")
# Import utilities
from utilities import KernelNN, KernelNN_subgraph, setup_device, get_output_affinities



In [ ]:
# choose data
model_type = 'subgraph'


# ---------------------------------------------------------------------------
sys.path.append(f"{project_root}/dMRI/scripts/")
from utilities_dMRI import dMRI_Graph_Generator as Data_Generator


# Paths
trained_models_path = project_root / 'dMRI' / 'trained_models' / model_type
training_logs_path = trained_models_path / 'logs'
data_path = project_root.parent / 'dMRI' / 'data'  # Adjust if needed

print('Available models:\n')
files = sorted(os.listdir(trained_models_path), reverse=True)

for i, f in enumerate(files):
    if Path(f).suffix == ".pth":
        print(f'[{i}] {f}')


In [ ]:
# Set model ID
model_id = 'base-subgraph-2026-04-04_t23-11-40'



# ---------------------------------------------------------------------------
if model_id.endswith('.pth'):
    model_id = model_id[:-4]

# Load full configuration (includes model, dataset, and training configs)
config_path = f'{training_logs_path}/configs/{model_id}.json'
with open(config_path, 'r') as f:
    config = json.load(f)

# Load loss records
loss_df = pd.read_csv(f'{training_logs_path}/loss_records/{model_id}.csv')

print('Configuration:')
for key, value in config.items():
    print(f'  {key}: {value}')

print(f'\Training completed {len(loss_df)} epochs')
print(f'Final training loss: {loss_df["loss"].iloc[-1]:.4f}')
print(f'Final validation loss: {loss_df["val_loss"].iloc[-1]:.4f}')

In [ ]:
# Plot training history
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(loss_df['epoch'], loss_df['loss'], label='Training Loss', marker='o')
ax.plot(loss_df['epoch'], loss_df['val_loss'], label='Validation Loss', marker='s')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title(f'Training History - {model_id}')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
model = KernelNN_subgraph(
    width=config['width'],
    ker_width=config['ker_width'],
    depth=config['depth'],
    ker_in=config['edge_attr'],  
    known_node_dim=config['input_node_features'],  
    unknown_node_dim=config['output_node_features'],
    out_width=config['out_width']
).to(device)


# Load trained weights
model_path = f'{trained_models_path}/{model_id}.pth'
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

print(f'Model loaded: {model_id}')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
globals().update(config)

In [ ]:
Generator = Data_Generator(data_path)
Generator.load_data('../data/100206/T1w/Diffusion/')

In [ ]:
((Generator.normed_coords[:,0]  < 0.5) & (Generator.normed_coords[:,0]  > 0.45))

In [ ]:
#select = torch.all((Generator.normed_coords < 0.75) & (Generator.normed_coords > 0.25),dim=1)
select = ((Generator.normed_coords[:,0]  < 0.5) & (Generator.normed_coords[:,0]  > 0.45))
select.sum().item()


In [ ]:
Generator.da = Generator.da[select] 
Generator.normed_coords = Generator.normed_coords[select] 

In [ ]:
out_graph = Generator.sample_graph(N=40000, k=config["k_out"], perplexity=config["perplexity"], sampling_method=config["output_sampling_method"], gen_feature_mode=config["feature_mode"])
in_graph = Generator.sample_graph(N=10000, k=config["k_in"], perplexity=config["perplexity"], sampling_method=config["input_sampling_method"], gen_feature_mode=config["feature_mode"])
cross_graph = Generator.connect_graphs(in_graph=in_graph, out_graph=out_graph, k_cross=config["k_cross"])
sample = cross_graph.to(device)


print('predict results')
output = model(sample).detach().cpu().numpy()

In [ ]:
coords = cross_graph.target.normed_coords.detach().cpu().numpy()
p1 = np.percentile(output, 1, axis=0)
p99 = np.percentile(output, 99, axis=0)

eps = 1e-8
RGB_brain = (output - p1) / (p99 - p1 + eps)

RGB_brain = np.clip(RGB_brain, 0, 1)

In [ ]:
viewer = napari.Viewer(ndisplay=3)
viewer.add_points(coords, face_color=RGB_brain, size=0.02)

In [ ]:
from scipy.interpolate import griddata

grid_res = 300  # or higher for more detail
x = np.linspace(0, 1, grid_res)
y = np.linspace(0, 1, grid_res)
z = np.linspace(0, 1, grid_res)
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
grid_points = np.vstack([X.ravel(), Y.ravel(), Z.ravel()]).T

# Interpolate each channel
R = griddata(coords, RGB_brain[:, 0], grid_points, method='linear', fill_value=0).reshape(grid_res, grid_res, grid_res)
G = griddata(coords, RGB_brain[:, 1], grid_points, method='linear', fill_value=0).reshape(grid_res, grid_res, grid_res)
B = griddata(coords, RGB_brain[:, 2], grid_points, method='linear', fill_value=0).reshape(grid_res, grid_res, grid_res)

RGB_grid = np.stack([R, G, B], axis=-1)

viewer = napari.Viewer(ndisplay=3)
viewer.add_image(RGB_grid, rgb=True)